# Lab - BUILD: Structured Return Requests

The last two labs built the pieces. This one puts them together into the module
ShopAssist actually uses: a messy sentence goes in, a decision your backend can act on
comes out.

| Task | What you learn |
| --- | --- |
| 1 | Extraction and validation are one call, in a fixed order |
| 2 | The extraction can raise a hand your rules did not think to raise |
| 3 | Five real messages, and the number a support team watches |

**You do not need an API key.** This lab ships with a Claude simulator, so every cell
runs offline. The code you write is exactly the code you would write against the real
API - set `ANTHROPIC_API_KEY` at home and the same notebook calls Claude for real.

**Almost everything here is already written.** The schema, the extractor and the three
checks are provided in the setup cell - they are the code you completed in the previous
two labs. What is left is the assembly, which is the actual subject of this lesson.

## Setup

Run this cell first, before anything else. Click it, then press **Shift+Enter**.

It is long, and none of it is new. Skim it: the schema, `extract_return_request()` with
its forced tool call, and the three checks - `schema_errors`, `missing_fields`,
`needs_human_review`. One thing did change: the schema now also asks for `confidence`,
`conflict_detected` and `human_review_required`, which Task 2 uses.

In [ ]:
# --- Lab setup (provided - just run it) ---
import json
import shopassist_lab
from shopassist_lab import check

# Everything below this line is ordinary Claude API code.
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()

model = "claude-sonnet-4-6"

ALLOWED_ACTIONS = ["refund", "replacement", "store_credit", "unclear"]
CONFIDENCE_THRESHOLD = 0.7
MUST_BE_PRESENT = ["order_id", "item"]

FIELDS = [
    "order_id", "item", "reason", "reason_detail", "desired_action",
    "desired_action_detail", "evidence_provided", "urgency",
    "missing_information", "confidence", "conflict_detected",
    "conflict_reason", "human_review_required",
]

# --- Step 1: the schema (from the structured output lab, plus three new fields)
tools = [
    {
        "name": "extract_return_request",
        "description": "Extract a structured return request from a customer support message.",
        "input_schema": {
            "type": "object",
            "properties": {
                "order_id": {"type": ["string", "null"]},
                "item": {"type": ["string", "null"]},
                "reason": {
                    "type": "string",
                    "enum": ["normal_return", "damaged_item", "billing_dispute",
                             "policy_exception", "unclear", "other"],
                },
                "reason_detail": {"type": ["string", "null"]},
                "desired_action": {
                    "type": "string",
                    "enum": ["refund", "replacement", "store_credit", "unclear", "other"],
                },
                "desired_action_detail": {"type": ["string", "null"]},
                "evidence_provided": {"type": "boolean"},
                "urgency": {"type": "string", "enum": ["low", "normal", "high", "unclear"]},
                "missing_information": {"type": "array", "items": {"type": "string"}},
                # New in this lab:
                "confidence": {"type": "number"},
                "conflict_detected": {"type": "boolean"},
                "conflict_reason": {"type": ["string", "null"]},
                # This one lets the extraction step tell the rest of the system
                # "a person should look at this" - see Task 2.
                "human_review_required": {"type": "boolean"},
            },
            "required": FIELDS,
        },
    }
]


# --- Steps 2 and 3: forced tool choice, then read the tool_use block
def extract_return_request(customer_message):
    prompt = f"""
Extract the return request from this customer message.

Customer message:
{customer_message}
"""
    message = client.messages.create(
        model=model,
        max_tokens=600,
        tools=tools,
        tool_choice={"type": "tool", "name": "extract_return_request"},
        messages=[{"role": "user", "content": prompt}],
    )
    tool_use = next(block for block in message.content if block.type == "tool_use")
    return tool_use.input


# --- Steps 4 to 6: the checks you wrote in the validation lab
def schema_errors(data):
    errors = []
    if data["desired_action"] not in ALLOWED_ACTIONS:
        errors.append("desired_action is {!r}, which is not allowed".format(
            data["desired_action"]))
    return errors


def missing_fields(data):
    gaps = []
    for field in MUST_BE_PRESENT:
        if data[field] is None:
            gaps.append(field)
    return gaps


def needs_human_review(data):
    if data["conflict_detected"]:
        return True
    if data["confidence"] < CONFIDENCE_THRESHOLD:
        return True
    return False

---

## Task 1 - Wire extraction and validation into one call

In [ ]:
# ============================================================
# TASK 1 - Wire extraction and validation into one call
# ============================================================
#
# WHAT TO DO
#   Write the function that takes a customer message and gives
#   back both the extracted data and the decision about what
#   happens to it.
#
# WHY IT MATTERS
#   Everything inside this function already exists. What did not
#   exist until now is the ORDER, and the order is the design.
#
#   Extract first, because there is nothing to check until
#   Claude has returned something. Then the value our system
#   cannot use, because that is worth one retry. Then the field
#   the customer never gave, because that needs a question, not
#   a retry. Only then how risky the case is.
#
#   Get the order wrong and you send a retry for a missing order
#   number - which is how systems learn to invent them.
#
# WHERE TO SEE IT IN THE LECTURE
#   "Step 1. Define the extraction schema" - about 1 minute
#   40 seconds in, and the five steps that follow it.
#
# HOW TO DO IT
#   Four blanks, and every name already exists in the setup
#   cell:
#
#       extract_return_request(customer_message)
#       schema_errors(data)
#       missing_fields(data)
#       needs_human_review(data)
# ============================================================

def process(customer_message):
    # BEGIN SOLUTION
    data = extract_return_request(customer_message)

    if schema_errors(data):
        decision = "retry with feedback"
    elif missing_fields(data):
        decision = "ask the customer"
    elif needs_human_review(data):
        decision = "human review"
    else:
        decision = "automate"
    # SCAFFOLD: data = ...   # extract_return_request(customer_message) - the forced tool call
    # SCAFFOLD:
    # SCAFFOLD: if ...:   # schema_errors(data) - a value our policy does not support
    # SCAFFOLD:     decision = "retry with feedback"
    # SCAFFOLD: elif ...:   # missing_fields(data) - something the customer never said
    # SCAFFOLD:     decision = "ask the customer"
    # SCAFFOLD: elif ...:   # needs_human_review(data) - a conflict, or low confidence
    # SCAFFOLD:     decision = "human review"
    # SCAFFOLD: else:
    # SCAFFOLD:     decision = "automate"
    # END SOLUTION: replace each ... below with the value named beside it

    return {"data": data, "decision": decision}


result = process("I want to return order ORD-12345678. The headphones arrived broken.")

# Provided: turns a leftover blank into a sentence.
if result["data"] is Ellipsis:
    raise ValueError("You still have ... above. Replace each one with the call named "
                     "in the comment beside it.")

print(json.dumps(result["data"], indent=2))
print()
print("decision:", result["decision"])

check("pipeline", process=process)

---

## Task 2 - Honour the flag the extraction itself raised

In [ ]:
# ============================================================
# TASK 2 - Honour the flag the extraction itself raised
# ============================================================
#
# WHAT TO DO
#   Widen the review decision so that the extraction's own
#   human_review_required flag is enough on its own.
#
# WHY IT MATTERS
#   Run the cell and read the comparison it prints. The customer
#   bought something six months ago and wants a refund. There is
#   no contradiction, and confidence is 0.78 - above your
#   threshold. Your own checks let it straight through.
#
#   The extraction flagged it anyway, because a purchase from
#   six months ago is a policy question rather than a return.
#
#   So the model can raise a hand your rules did not think to
#   raise. What it cannot do is lower one: your checks still
#   run, and either side is enough to stop the case. Trust the
#   flag upward, never downward.
#
# WHERE TO SEE IT IN THE LECTURE
#   "Step 6. Human review routing" - about 4 minutes 37 seconds
#   in. The field itself is described at 2 minutes 21 seconds.
#
# HOW TO DO IT
#   One blank: the flag the extraction set, which lives in the
#   data under the key "human_review_required".
# ============================================================

def review_required(data):
    # Our own checks, from the previous lab.
    if needs_human_review(data):
        return True

    # BEGIN SOLUTION
    if data["human_review_required"]:
        return True
    # SCAFFOLD: if ...:   # data["human_review_required"] - the flag the extraction set
    # SCAFFOLD:     return True
    # END SOLUTION: replace the ... below with the value named beside it

    return False


# Provided: rebuild process() so it uses the wider check.
def process(customer_message):
    data = extract_return_request(customer_message)

    if schema_errors(data):
        decision = "retry with feedback"
    elif missing_fields(data):
        decision = "ask the customer"
    elif review_required(data):
        decision = "human review"
    else:
        decision = "automate"

    return {"data": data, "decision": decision}


POLICY_CASE = ("I bought this camera six months ago, but I still want a refund "
               "because I never used it. Order ORD-44445555.")

policy_data = extract_return_request(POLICY_CASE)

print("reason:               ", policy_data["reason"])
print("confidence:           ", policy_data["confidence"], "(threshold is",
      str(CONFIDENCE_THRESHOLD) + ")")
print("conflict_detected:    ", policy_data["conflict_detected"])
print()
print("our checks alone say: ", needs_human_review(policy_data))
print("the extraction says:  ", policy_data["human_review_required"])
print("together:             ", review_required(policy_data))

check("model_flag", review_required=review_required)

---

## Task 3 - Run five real messages through it

In [ ]:
# ============================================================
# TASK 3 - Run five real messages through it
# ============================================================
#
# WHAT TO DO
#   Put five different customer messages through the finished
#   module, collect what happened to each, and work out what
#   share of them went through without a person.
#
# WHY IT MATTERS
#   One message proves nothing - you learned that in the
#   evaluation lab, and it is just as true here.
#
#   Read the table this prints. Five messages, and each stops
#   for a different, nameable reason: a missing order number, a
#   purchase outside the policy, a message that contradicts
#   itself. None of them stopped because "the model failed".
#
#   The last number is the one a support team actually watches.
#   Push it up by improving the prompt and the schema. Never
#   push it up by loosening the checks - a system that automates
#   everything is a system with no idea when it is wrong.
#
# WHERE TO SEE IT IN THE LECTURE
#   "For the demo, we can run a few realistic messages" - about
#   5 minutes 3 seconds in.
#
# HOW TO DO IT
#   One blank: the module you finished in Task 1, called on
#   each message in turn.
#
#       process(customer_message)
# ============================================================

INBOX = [
    "I want to return order ORD-12345678. The headphones arrived broken.",
    "I bought a jacket last week and want to return it. I do not have the order number.",
    "This keyboard is not what I expected. Can I get my money back? Order ORD-22223333.",
    "I bought this camera six months ago, but I still want a refund because I never used it. Order ORD-44445555.",
    "The speaker in order ORD-66667777 works perfectly, but it arrived broken and I need a replacement today.",
]

results = []

for customer_message in INBOX:
    # BEGIN SOLUTION
    outcome = process(customer_message)
    # SCAFFOLD: outcome = ...   # process(customer_message) - the module from Task 1
    # END SOLUTION: replace the ... below with the value named beside it

    # Provided: turns a leftover blank into a sentence.
    if outcome is Ellipsis:
        raise ValueError("You still have ... above. Replace it with "
                         "process(customer_message).")

    results.append({
        "message": customer_message,
        "data": outcome["data"],
        "decision": outcome["decision"],
    })

# Provided: the share that needed no person at all.
automated = [row for row in results if row["decision"] == "automate"]
automation_rate = len(automated) / len(results)

print("{:<22} {:<18} {}".format("decision", "reason", "why it stopped"))
for row in results:
    data = row["data"]
    why = (schema_errors(data) or missing_fields(data)
           or data["conflict_reason"]
           or ("policy question" if data["reason"] == "policy_exception" else None)
           or ("confidence {}".format(data["confidence"])
               if data["confidence"] < CONFIDENCE_THRESHOLD else "-"))
    print("{:<22} {:<18} {}".format(row["decision"], data["reason"], why))

print()
print("Automated without a person: {} of {}  ({:.0%})".format(
    len(automated), len(results), automation_rate))

check("inbox", results=results, automation_rate=automation_rate)

---

## Done

That table is the whole of Section 3 in one screen.

A customer types a sentence. A schema turns it into fields your code can read. Checks
that never talk to a model decide whether those fields can be acted on. And what comes
out is not an answer - it is a **decision**: automate this one, ask about that one, put
a person on these two.

Nothing here relies on prompting alone, which is the point the exam keeps testing. Use
tool use and a schema to get structure. Use nullable fields so nothing gets invented.
Use enums to keep categories to the ones you can route. Validate in code. Retry only
what is fixable. Route the rest to a person.

ShopAssist can now understand what a customer wants. In the next section it starts doing
something about it - looking up orders, checking policy, processing refunds and
escalating what it should not decide alone.